In [5]:
import json
import re
from pathlib import Path
import datetime as dt
import pyarrow

import pandas as pd


START_DATE = dt.date(2023, 1, 1)
END_DATE   = dt.date(2026, 1, 1)

KW = re.compile(r"(?i)\b(tsla|\$tsla|tesla)\b")

IN_PATH  = Path("../data/equity_data/stocks_submissions")
OUT_PATH = Path("../data/equity_data/tesla_stocks_posts.parquet")


def read_ndjson_plain(path: Path):
    # errors="ignore" bo czasem trafią się dziwne znaki
    with path.open("rt", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue
            yield obj


def to_utc_date(created_utc: int) -> dt.date:
    return dt.datetime.utcfromtimestamp(int(created_utc)).date()


rows = []
i = 0
matched = 0

for obj in read_ndjson_plain(IN_PATH):
    i += 1
    created_utc = obj.get("created_utc")
    if created_utc is None:
        continue

    d = to_utc_date(created_utc)
    if d < START_DATE or d >= END_DATE:
        continue

    title = obj.get("title") or ""
    selftext = obj.get("selftext") or ""
    text = f"{title}\n{selftext}"

    if not KW.search(text):
        continue

    matched += 1
    rows.append({
        "id": obj.get("id"),
        "date_utc": d.isoformat(),
        "created_utc": int(created_utc),
        "subreddit": obj.get("subreddit"),
        "title": title,
        "selftext": selftext,
        "score": obj.get("score"),
        "num_comments": obj.get("num_comments"),
        "permalink": obj.get("permalink"),
        "url": obj.get("url"),
    })

    if matched % 50_000 == 0:
        print(f"Znaleziono: {matched:,} (przetworzono linii: {i:,})")

df = pd.DataFrame(rows)
df.to_parquet(OUT_PATH, index=False)
print("Gotowe ->", OUT_PATH.resolve(), " | rekordów:", len(df))

Gotowe -> C:\Users\user\OneDrive\Documents\magisterka\praca magisterska\code\equity_data_importers\data\tesla_stocks_posts.parquet  | rekordów: 3406


In [6]:
import pandas as pd

df = pd.read_parquet("../data/equity_data/tesla_stocks_posts.parquet")
df["date_utc"] = pd.to_datetime(df["date_utc"])

print("min date:", df["date_utc"].min())
print("max date:", df["date_utc"].max())

print(df["date_utc"].dt.year.value_counts().sort_index())

min date: 2023-01-01 00:00:00
max date: 2025-12-31 00:00:00
date_utc
2023     753
2024    1007
2025    1646
Name: count, dtype: int64


In [7]:
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

START = "2023-01-01"
END   = "2025-12-31"   # włącznie

df = pd.read_parquet("../data/equity_data/tesla_stocks_posts.parquet")
df["date"] = pd.to_datetime(df["date_utc"]).dt.strftime("%Y-%m-%d")

an = SentimentIntensityAnalyzer()
df["text"] = df["title"].fillna("") + "\n" + df["selftext"].fillna("")
df["sentiment"] = df["text"].map(lambda t: an.polarity_scores(t)["compound"])

daily = (
    df.groupby("date", as_index=False)
      .agg(
          reddit_posts=("id", "count"),
          reddit_sent_mean=("sentiment", "mean"),
          reddit_sent_sum=("sentiment", "sum"),
          reddit_score_sum=("score", "sum"),
          reddit_comments_sum=("num_comments", "sum"),
      )
)

calendar = pd.DataFrame({"date": pd.date_range(START, END, freq="D").strftime("%Y-%m-%d")})

daily_full = calendar.merge(daily, on="date", how="left")

# dni bez postów -> 0 dla countów/sum, a średnia sentymentu zostaje NaN (albo też 0 jeśli wolisz)
daily_full["reddit_posts"] = daily_full["reddit_posts"].fillna(0).astype(int)
daily_full["reddit_score_sum"] = daily_full["reddit_score_sum"].fillna(0)
daily_full["reddit_comments_sum"] = daily_full["reddit_comments_sum"].fillna(0)
daily_full["reddit_sent_sum"] = daily_full["reddit_sent_sum"].fillna(0)

daily_full.to_csv("../data/equity_data/stock-reddit-data.csv", index=False)
print("Zapisano -> stock-reddit-data.csv", "dni:", len(daily_full))

Zapisano -> stock-reddit-data.csv dni: 1096
